In [ ]:
from data.dataset import DOGVideoREIDDataset
from configs.config import Config
from data.transforms import ViTVideoTransform
from torch.utils.data import DataLoader
from samplers.sampler import RandomIdentitySampler
cfg = Config()

In [12]:

train_dataset = DOGVideoREIDDataset(
    root_dir=cfg.data_root,
    split_file=cfg.split_file,
    split="train",
    world=cfg.world,
    clip_len=cfg.clip_len,
    transform=ViTVideoTransform()
)

sampler = RandomIdentitySampler(
    train_dataset,
    num_ids=cfg.num_ids,          # e.g. 4
    num_instances=cfg.num_instances  # e.g. 2
)

query_dataset = DOGVideoREIDDataset(
    root_dir=cfg.data_root,
    split_file=cfg.split_file,
    split="query",
    world=cfg.world,
    clip_len=cfg.clip_len,
    transform=ViTVideoTransform()
)

In [14]:
print("Dataset size:", len(train_dataset))

clip, label, dog_id, video_id = train_dataset[0]

print("Clip shape:", clip.shape)
print("Label:", label)
print("Dog ID:", dog_id)
print("Video ID:", video_id)

Dataset size: 3756
Clip shape: torch.Size([1, 3, 224, 224])
Label: 0
Dog ID: 006e92b4-8c6a-4fb2-a870-d172b70546a5
Video ID: 006e92b4-8c6a-4fb2-a870-d172b70546a5-666b589d-ead2-4f01-ab24-6ffe44c7b99f.mp4


In [15]:
print("Dataset size:", len(query_dataset))

clip, label, dog_id, video_id = query_dataset[0]

print("Clip shape:", clip.shape)
print("Label:", label)
print("Dog ID:", dog_id)
print("Video ID:", video_id)

Dataset size: 1594
Clip shape: torch.Size([1, 3, 224, 224])
Label: 0
Dog ID: 0062cada-a402-41bb-980e-6ae5e0672440
Video ID: 0062cada-a402-41bb-980e-6ae5e0672440-6efb5b41-a90d-4833-b72a-3840bb0e9298.mp4


In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,   # must = num_ids * num_instances
    sampler=sampler,
    num_workers=4,
    drop_last=True
)

query_loader = DataLoader(
    query_dataset,
    batch_size=cfg.batch_size,   # must = num_ids * num_instances
    sampler=sampler,
    num_workers=4,
    drop_last=True
)

In [8]:
batch = next(iter(train_loader))

clips, labels, dog_ids, video_ids = batch

print("Clips shape:", clips.shape)
print("Labels shape:", labels.shape)
print("Example labels:", labels[:5])

Clips shape: torch.Size([8, 1, 3, 224, 224])
Labels shape: torch.Size([8])
Example labels: tensor([189, 189, 619, 619, 609])


In [16]:
clips, labels, dog_ids, video_ids = next(iter(train_loader))

print("Batch clips:", clips.shape)
print("Batch labels:", labels.shape)

Batch clips: torch.Size([8, 1, 3, 224, 224])
Batch labels: torch.Size([8])


In [17]:
import torch

def compute_similarity_matrix(self, qf, gf):

    qf = torch.nn.functional.normalize(qf, dim=1)
    gf = torch.nn.functional.normalize(gf, dim=1)

    sim_mat = qf @ gf.T

    return sim_mat


def compute_metrics(self, similarity_mat, query_labels, gallery_labels):

    query_labels = torch.tensor(query_labels)
    gallery_labels = torch.tensor(gallery_labels)

    num_queries = len(query_labels)
    num_gallery = len(gallery_labels)

    cmc_curve = torch.zeros(num_gallery)

    ap_list = []

    print(similarity_mat.shape)
    print(len(query_labels), len(gallery_labels))

    for query_label, similarity_row in zip(query_labels, similarity_mat):

        sorted_indices = torch.argsort(similarity_row, descending=True)

        matches = (gallery_labels[sorted_indices] == query_label).float()

        if matches.sum() == 0:
            continue

        rank = matches.nonzero(as_tuple=False)[0].item()

        cmc_curve[rank:] += 1

        cum_matches = matches.cumsum(0)

        precision_at_k = cum_matches / torch.arange(
            1, num_gallery + 1, dtype=torch.float32
        )

        ap = (precision_at_k * matches).sum() / matches.sum()

        ap_list.append(ap)

    cmc_curve = cmc_curve / num_queries

    mAP = torch.stack(ap_list).mean().item()

    return cmc_curve, mAP

In [20]:
import torch

# 3 queries, 5 gallery items
query_labels = [0, 1, 2]
gallery_labels = [0, 1, 2, 3, 4]

# fake similarity (force correct ranking)
similarity_mat = torch.tensor([
    [0.9, 0.1, 0.2, 0.0, 0.0],  # query 0 → best match index 0
    [0.1, 0.9, 0.2, 0.0, 0.0],  # query 1 → index 1
    [0.1, 0.9, 0.4, 0.0, 0.0],  # query 2 → index 2
])

cmc, mAP = compute_metrics(None, similarity_mat, query_labels, gallery_labels)

print("CMC:", cmc)
print("mAP:", mAP)

torch.Size([3, 5])
3 5
CMC: tensor([0.6667, 1.0000, 1.0000, 1.0000, 1.0000])
mAP: 0.8333333134651184
